# Best Buy Printer Reviews Scraper

Scrapes up to 100 of the most recent reviews for each of 30 printers.


In [ ]:
import time, csv, math
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ── ChromeDriver setup ────────────────────────────────────────────────
options = Options()
options.add_argument("--lang=en-US")
# options.add_argument("--headless")   # uncomment to hide browser window

service = Service(r"C:\Users\yizha\Downloads\chromedriver-win64\chromedriver.exe")  # ← your path here
driver  = webdriver.Chrome(service=service, options=options)
wait    = WebDriverWait(driver, 10)

# ── CSS selector for the review text only ───────────────────────────────
CSS_TEXT = "div.ugc-review-body > p"


In [ ]:
review_urls = [
    "https://www.bestbuy.com/site/reviews/epson-expression-home-xp-4200-all-in-one-inkjet-printer-black/6516132",
    "https://www.bestbuy.com/site/reviews/hp-smart-tank-5101-wireless-all-in-one-supertank-inkjet-printer-with-up-to-2-years-of-ink-included-white/6527565",
    "https://www.bestbuy.com/site/reviews/hp-envy-inspire-7255e-wireless-all-in-one-inkjet-photo-printer-with-3-months-of-instant-ink-included-with-hp-white-sandstone/6492187",
    "https://www.bestbuy.com/site/reviews/hp-deskjet-2855e-wireless-all-in-one-inkjet-printer-with-3-months-of-instant-ink-included-with-hp-white/6574145",
    "https://www.bestbuy.com/site/reviews/canon-pixma-tr4720-wireless-all-in-one-inkjet-printer-black/6477031",
    "https://www.bestbuy.com/site/reviews/hp-officejet-pro-9125e-wireless-all-in-one-inkjet-printer-with-3-months-of-instant-ink-included-with-hp-white/6565475",
    "https://www.bestbuy.com/site/reviews/hp-officejet-8015e-wireless-all-in-one-inkjet-printer-with-6-months-of-instant-ink-included-with-hp-white/6529738",
    "https://www.bestbuy.com/site/reviews/epson-ecotank-et-3830-all-in-one-supertank-inkjet-printer-white/6470012",
    "https://www.bestbuy.com/site/reviews/canon-pixma-ts6420a-wireless-all-in-one-inkjet-printer-black/6494255",
    "https://www.bestbuy.com/site/reviews/hp-envy-inspire-7955e-wireless-all-in-one-inkjet-photo-printer-with-3-months-of-instant-ink-included-with-hp-white-sandstone/6478251",
    "https://www.bestbuy.com/site/reviews/hp-envy-6155e-wireless-all-in-one-inkjet-printer-with-3-months-of-instant-ink-included-with-hp-white/6590093",
    "https://www.bestbuy.com/site/reviews/epson-ecotank-et-4850-all-in-one-supertank-inkjet-printer-white/6470011",
    "https://www.bestbuy.com/site/reviews/epson-ecotank-et-2800-wireless-all-in-one-supertank-inkjet-printer-white/6469371",
    "https://www.bestbuy.com/site/reviews/canon-pixma-megatank-g3270-wireless-all-in-one-supertank-inkjet-printer-black/6535601",
    "https://www.bestbuy.com/site/reviews/hp-envy-6555e-wireless-all-in-one-inkjet-printer-with-3-months-of-instant-ink-included-with-hp-white/6590105",
    "https://www.bestbuy.com/site/reviews/brother-mfc-j1010dw-wireless-color-all-in-one-refresh-subscription-eligible-inkjet-printer-black/6503815",
    "https://www.bestbuy.com/site/reviews/epson-ecotank-photo-et-8550-all-in-one-wide-format-supertank-printer-white/6469372",
    "https://www.bestbuy.com/site/reviews/hp-officejet-pro-8135e-wireless-all-in-one-inkjet-printer-with-3-months-of-instant-ink-included-with-hp-white/6565476",
    "https://www.bestbuy.com/site/reviews/hp-officejet-pro-9135e-wireless-all-in-one-inkjet-printer-with-3-months-of-instant-ink-included-with-hp-white/6565473",
    "https://www.bestbuy.com/site/reviews/canon-pixma-tr8620a-wireless-all-in-one-inkjet-printer-with-fax-black/6494259",
    "https://www.bestbuy.com/site/reviews/epson-ecotank-et-3850-all-in-one-supertank-inkjet-printer-white/6470010",
    "https://www.bestbuy.com/site/reviews/hp-smart-tank-6001-wireless-all-in-one-supertank-inkjet-printer-with-up-to-2-years-of-ink-included-basalt/6498188",
    "https://www.bestbuy.com/site/reviews/epson-ecotank-et-3850-all-in-one-cartridge-free-supertank-printer-refurbished-white/6589868",
    "https://www.bestbuy.com/site/reviews/canon-pixma-ts7720-wireless-all-in-one-inkjet-printer-white/6561810",
    "https://www.bestbuy.com/site/reviews/hp-smart-tank-7602-wireless-all-in-one-supertank-inkjet-printer-with-up-to-2-years-of-ink-included-dark-surf-blue/6498190",
    "https://www.bestbuy.com/site/reviews/epson-ecotank-et-2400-wireless-color-all-in-one-cartridge-free-supertank-printer-with-scan-and-copy-black/6525631",
    "https://www.bestbuy.com/site/reviews/hp-deskjet-4255e-wireless-all-in-one-inkjet-printer-with-3-months-of-instant-ink-included-with-hp-white/6575024",
    "https://www.bestbuy.com/site/reviews/epson-ecotank-pro-et-5850-wireless-all-in-one-inkjet-printer-white/6397580",
    "https://www.bestbuy.com/site/reviews/hp-laserjet-pro-mfp-3301sdw-wireless-color-all-in-one-laser-printer-white/6582284",
    "https://www.bestbuy.com/site/reviews/epson-ecotank-et-15000-wireless-all-in-one-inkjet-printer-white/6397577",
]

In [ ]:
with open("printer_reviews.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    # add product_name column
    writer.writerow(["product_url", "product_name", "review_text"])

    for idx, url in enumerate(review_urls, 1):
        # extract slug after /reviews/ and before next /
        product_name = url.split("/reviews/")[1].split("/")[0]
        print(f"[{idx}/{len(review_urls)}] Scraping {product_name}")

        for page in range(0, math.ceil(100/20) + 1):
            driver.get(f"{url}?sort=MOST_RECENT&page={page}")
            time.sleep(2)  # let JS render

            paras = driver.find_elements(By.CSS_SELECTOR, CSS_TEXT)
            if not paras:
                break
            print(f"  • Found {len(paras)} reviews on page {page}")

            for p in paras:
                text = p.text.strip().replace("\u2019", "'")
                # write URL, product_name, and review text
                writer.writerow([url, product_name, text])

            if len(paras) < 10:
                break

In [ ]:
driver.quit()
print("✅ Done! Reviews saved to printer_reviews.csv")


# Unsupervised aspect extraction

## LDA

In [ ]:
import re
import string
import emoji
import pandas as pd
import nltk

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords, wordnet
from textblob import TextBlob
import spacy
from cleantext import clean, clean_words

from gensim import corpora
from gensim.models.ldamulticore import LdaMulticore
from bertopic import BERTopic

In [ ]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

stop_words = set(stopwords.words("english"))

In [ ]:
class AntonymReplacer:
    def replace(self, word, pos=None):
        antonyms = set()
        for syn in wordnet.synsets(word, pos=pos):
            for lemma in syn.lemmas():
                for ant in lemma.antonyms():
                    antonyms.add(ant.name())
        return antonyms.pop() if len(antonyms)==1 else None

    def replace_negations(self, sent):
        i, l, out = 0, len(sent), []
        while i < l:
            w = sent[i]
            if w == "not" and i+1<l:
                ant = self.replace(sent[i+1])
                if ant:
                    out.append(ant); i+=2; continue
            out.append(w); i+=1
        return out

# Deep preprocessing function
def preprocess(text):
    # 1) Basic TextBlob cleanup (drops hashtags, normalizes)
    text = " ".join(TextBlob(text).words)
    # 2) Remove digits
    text = re.sub(r"\d+", "", text)
    # 3) Lowercase
    text = text.lower()
    # 4) Emoji → text
    text = emoji.demojize(text)
    # 5) Strip punctuation
    for p in string.punctuation:
        text = text.replace(p, "")
    # 6) Tokenize
    tokens = word_tokenize(text)
    # 7) Keep only alphabetic & non-empty
    tokens = [t for t in tokens if t.isalpha()]
    # 8) Negation handling
    tokens = AntonymReplacer().replace_negations(tokens)
    # 9) Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    # 10) Lemmatize (only nouns, adj, verbs, adv)
    doc = nlp(" ".join(tokens))
    return [tok.lemma_ for tok in doc if tok.pos_ in {"NOUN","ADJ","VERB","ADV"}]

In [ ]:
# Load scraped reviews
df = pd.read_csv("printer_reviews.csv", encoding="utf-8-sig")
df['Clean'] = df['review_text'].astype(str).apply(lambda txt: preprocess(txt))
df.to_csv("preprocessed_reviews.csv", index=False)



In [ ]:
from ast import literal_eval
pre = pd.read_csv("preprocessed_reviews.csv")


In [ ]:
data_words = [literal_eval(x) for x in pre['Clean']]

# Build dictionary & corpus
id2word = corpora.Dictionary(data_words)
texts  = data_words
id2word.filter_extremes(no_below=5, no_above=0.5)
corpus = [id2word.doc2bow(text) for text in texts]

# Train LDA
num_topics = 50
num_words = 10 
lda = LdaMulticore(
    corpus=corpus,
    id2word=id2word,
    num_topics=num_topics,
    passes=10,
    workers=2,
    random_state=42
)


In [ ]:
# Print top terms per topic
print("LDA Topics:")
for tid, terms in lda.print_topics(num_topics=num_topics, num_words=num_words):
    print(f"Topic {tid}: {terms}")

In [ ]:
from pprint import pprint
pprint(lda.print_topics(num_topics=50, num_words=10))


In [ ]:
import os
import pickle
import pandas as pd
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

In [ ]:
out_dir = r"C:\Users\yizha\OneDrive\AIT303 Advanced Issues of AI (NLP) 202504 Chua Chong Chai\Assignment 1\Code\Task 2"

# Save topic terms into Excel 
topics = lda.print_topics(num_topics=num_topics, num_words=num_words)
df_topics = pd.DataFrame([t[1] for t in topics], columns=["topics"])
df_topics.to_csv(os.path.join(out_dir, "LDA_topics.csv"), index=False)

In [ ]:
# Dump model, dictionary & corpus
pickle.dump(lda, open(os.path.join(out_dir, "LDA_model.model"),   "wb"))
pickle.dump(id2word, open(os.path.join(out_dir, "LDA_dictionary.dict"), "wb"))
pickle.dump(corpus, open(os.path.join(out_dir, "LDA_corpus.corpus"),   "wb"))

In [ ]:
print(f"✅ Saved LDA topics, model, dictionary & corpus to `{out_dir}`")


In [ ]:
# Reload & prepare pyLDAvis

loaded_lda  = pickle.load(open(os.path.join(out_dir, "LDA_model.model"),   "rb"))
loaded_dict = pickle.load(open(os.path.join(out_dir, "LDA_dictionary.dict"), "rb"))
loaded_corp = pickle.load(open(os.path.join(out_dir, "LDA_corpus.corpus"),   "rb"))

In [ ]:
pyLDAvis.enable_notebook()
vis = gensimvis.prepare(loaded_lda, loaded_corp, loaded_dict)
pyLDAvis.display(vis)

# BERT

In [ ]:
# Prepare string docs for BERTopic
docs = [" ".join(literal_eval(x)) for x in pre['Clean']]

In [ ]:
# Train BERTopic
topic_model = BERTopic(language="english", verbose=True)
topics, probs = topic_model.fit_transform(docs)


In [ ]:
# View topic summary
info = topic_model.get_topic_info()
print(info.head(10))

In [ ]:
import pickle

path = r"C:\Users\yizha\OneDrive\AIT303 Advanced Issues of AI (NLP) 202504 Chua Chong Chai\Assignment 1\Code\Task 2"

topic_info = topic_model.get_topic_info()
topic_info.head(100).to_csv(f"{path}/BERTopic_topics.csv", index=False)

topic_model.save(f"{path}/BERTopic_model")


In [ ]:
for i in range(10):
    print(f"\nTop terms for topic {i}:")
    print(topic_model.get_topic(i))

fig1 = topic_model.visualize_topics()
fig2 = topic_model.visualize_hierarchy(top_n_topics=50)
fig3 = topic_model.visualize_barchart(top_n_topics=50)

# Display them
fig1.show()
fig2.show()
fig3.show()

In [ ]:
# LDA topic terms
lda_topics = {tid: [w.split("*")[1].strip() 
                    for w in terms.split("+")] 
              for tid, terms in lda.print_topics(50, num_words=10)}

# BERTopic key terms
bt_info = topic_model.get_topic_info()
bertopic_topics = {}
for tid in bt_info.Topic.unique():
    if tid == -1: continue
    terms = [t for t, _ in topic_model.get_topic(tid)]
    bertopic_topics[tid] = terms


In [ ]:
print(lda_topics)

In [ ]:
print(bertopic_topics)

# COREX

In [ ]:
seed_topics = {
    # 1) Ease-of-Use & Connectivity
    "easeofuse_connectivity": [
        "easy", "setup", "install", "installation", "instruction",
        "connect", "wifi", "wireless", "bluetooth", "usb",
        "app", "pair", "quick", "iphone", "ipad", "laptop"
    ],

    # 2) Print Quality & Output
    "print_quality": [
        "print", "quality", "photo", "image", "picture",
        "color", "colour", "vivid", "bright", "sharp",
        "resolution", "professional", "clear", "excellent"
    ],

    # 3) Ink & Cost Efficiency
    "ink_cost": [
        "ink", "cartridge", "tank", "refill", "refillable",
        "save", "saving", "cost", "money", "subscription",
        "instant", "economical", "running", "longevity"
    ],

    # 4) Speed & Noise Performance
    "speed_noise_performance": [
        "fast", "speed", "quick", "responsive",
        "slow", "quiet", "noise", "noisy",
        "performance", "efficient", "efficiency"
    ],

    # 5) Paper Handling & Jam Reliability
    "paper_handling": [
        "paper", "jam", "tray", "feed", "feeder",
        "page", "double", "duplex", "load", "stuck",
        "error", "scan", "scanner", "copy", "document"
    ],

    # 6) Reliability & Support
    "reliability_support": [
        "service", "customer", "support", "warranty",
        "firmware", "update", "policy", "call", "store",
        "helpful", "durable", "lasting", "long", "reliable",
        "year", "replace", "replacement", "problem", "issue", "break"
    ]
}



In [ ]:
anchors = list(seed_topics.values())

In [ ]:
import pickle
import scipy.sparse as ss
from corextopic import corextopic as ct
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

In [ ]:
docs = [" ".join(literal_eval(x)) for x in pre['Clean']]

In [ ]:
vectorizer = CountVectorizer(
    stop_words="english",
    binary=True
)

X = vectorizer.fit_transform(docs)
X = ss.csr_matrix(X)
vocab = vectorizer.get_feature_names_out()

In [ ]:
model = ct.Corex(
    n_hidden=len(anchors),
    words=vocab,
    seed=42,
    max_iter=2000,
    verbose=True
)

model.fit(
    X,
    words=vocab,
    anchors=anchors,
    anchor_strength=3
)

In [ ]:
print("Anchored CorEx Aspects:")
topics = model.get_topics()
for i, name in enumerate(seed_topics):
    top_words = [tup[0] for tup in topics[i]]
    print(f"{name}: {top_words}")

In [ ]:
with open("corEx_aspect_model.pkl","wb") as f:
    pickle.dump(model, f)

df = pd.DataFrame(
    model.get_topics(n_words=12),
    columns=[f"Word_{j}" for j in range(1,13)]
)
df.insert(0, "Aspect", list(seed_topics.keys()))
df.to_excel("corEx_aspect_topics.xlsx", index=False)

print("\n✅ Anchored CorEx model and topics saved.")

#### Process and label all the downloaded product reviews using the trained CorEx model and sentiment analysis model

In [ ]:
import pandas as pd
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

CSV_PATH              = "preprocessed_reviews.csv"     
COREX_MODEL_PATH      = "corEx_aspect_model.pkl"            
SENTIMENT_MODEL_PATH  = "bilstm_skipgram.h5"                   
TOKENIZER_PATH        = "tokenizer.pkl"                       
MAX_LEN               = 300                                     

# load data
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

# load models
corex       = pickle.load(open(COREX_MODEL_PATH, "rb"))
sent_model  = load_model(SENTIMENT_MODEL_PATH)
tokenizer   = pickle.load(open(TOKENIZER_PATH, "rb"))


In [ ]:
# extract the 6 aspect word‐sets from Corex
topic_triples = corex.get_topics()          
aspect_sets   = [ {w for w, *_ in t} for t in topic_triples ]
aspect_sets   = aspect_sets[:6]             

# quick check
for i, s in enumerate(aspect_sets):
    print(f"Aspect {i} seeds:", sorted(list(s))[:10], "…")


In [ ]:
# Aspect Labeling & Sentiment Scoring 
# For each review row, we:
# 1. Tokenize the pre-cleaned text and build a set of unique tokens.
# 2. For each aspect’s keyword set, flag presence (1) if any keyword appears.
# 3. Reconstruct text from tokens and feed it to the sentiment model.
# 4. Store six binary aspect flags and the sentiment score back into `df`.

import ast
N = len(df)
labels = np.zeros((N, 6), dtype=int)   # placeholders for 6 aspect 
scores = np.zeros(N)                   # placeholder for sentiment scores

for i, row in df.iterrows():
    # Get the cleaned tokens
    toks = row["Clean"]
    if isinstance(toks, str):
        toks = ast.literal_eval(toks)
    ws = set(toks)

    
    # Check each aspect’s keyword set
    for j, aset in enumerate(aspect_sets):
        if ws & aset:                  
            labels[i, j] = 1

    # Prepare text for sentiment prediction
    text_for_model = " ".join(toks)
    seq = tokenizer.texts_to_sequences([text_for_model])
    pad = pad_sequences(
        seq,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )
    p = sent_model.predict(pad, verbose=0)
    scores[i] = p[0, 1] if p.shape[-1] == 2 else p[0, 0]

# Write results back to df
for j in range(6):
    df[f"aspect_{j}"] = labels[:, j]
df["sent_score"] = scores








In [ ]:
import matplotlib.pyplot as plt

# helper to truncate after "-all-in-one"
def shorten_name(name):
    marker = "-all-in-one"
    if marker in name:
        return name.split(marker)[0] + marker
    return name

aspect_names = [
    "EASE-OF-USE & CONNECTIVITY",
    "PRINT QUALITY & OUTPUT",
    "INK & COST EFFICIENCY",
    "SPEED & NOISE PERFORMANCE",
    "PAPER HANDLING & JAM RELIABILITY",
    "RELIABILITY & SUPPORT"
]


for j, aspect in enumerate(aspect_names):
    # filter + aggregate
    sub = df[df[f"aspect_{j}"] == 1]
    agg = sub.groupby("product_url")["sent_score"].sum()
    top5 = agg.nlargest(5)
    
    # print top-5 with full names
    print(f"\nTop 5 products for {aspect}:")
    entries = []
    for url, score in top5.items():
        pname = df.loc[df["product_url"] == url, "product_name"].iloc[0]
        print(f"   {score:6.3f} {pname} ({url})")
        entries.append((pname, score))
    
    # prepare shortened names and scores
    short_names = [shorten_name(n) for n, _ in entries]
    scores      = [s for _, s in entries]
    
    # plot horizontal bar chart
    plt.figure(figsize=(10, 4))
    plt.barh(short_names, scores, height=0.6)
    plt.gca().invert_yaxis()  # highest at top
    plt.xlabel("Total Positive Sentiment Score")
    plt.title(f"Top 5 Products for\n{aspect}")
    plt.tight_layout()
    plt.show()




